# 🚗 Task 3: Car Price Prediction with Machine Learning
**Dataset:** CarPrice.csv — 205 cars with features like brand, horsepower, mileage, engine size, body type, etc.

Target variable: `price`

In [ ]:
# !pip install scikit-learn pandas matplotlib seaborn

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded!")

## 2. Load the Real Dataset

In [ ]:
# Load directly from GitHub
url = "https://raw.githubusercontent.com/amankharwal/Website-data/master/CarPrice.csv"
df = pd.read_csv(url)

# OR local: df = pd.read_csv("CarPrice.csv")

print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print()
print("Price stats:")
print(df['price'].describe())

## 3. Data Cleaning & Feature Engineering

In [ ]:
# Extract brand name from CarName (first word)
df['Brand'] = df['CarName'].str.split(' ').str[0].str.lower()

# Fix brand typos
brand_fix = {
    'maxda': 'mazda', 'toyouta': 'toyota', 'vw': 'volkswagen',
    'vokswagen': 'volkswagen', 'porcshce': 'porsche'
}
df['Brand'] = df['Brand'].replace(brand_fix)

print("Unique brands:", sorted(df['Brand'].unique()))
print("Brand value counts:")
print(df['Brand'].value_counts())

In [ ]:
# Drop car_ID and CarName
df.drop(['car_ID', 'CarName'], axis=1, inplace=True)

# Encode categorical columns
cat_cols = ['fueltype','aspiration','doornumber','carbody','drivewheel',
            'enginelocation','enginetype','cylindernumber','fuelsystem','Brand']
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print("Final shape:", df.shape)
df.head()

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0,0].hist(df['price'], bins=30, color='#3498DB', edgecolor='white')
axes[0,0].set_title('Price Distribution', fontweight='bold'); axes[0,0].set_xlabel('Price ($)')

axes[0,1].hist(np.log1p(df['price']), bins=30, color='#E74C3C', edgecolor='white')
axes[0,1].set_title('Log(Price) Distribution', fontweight='bold')

axes[0,2].scatter(df['horsepower'], df['price'], alpha=0.5, color='#9B59B6', s=30)
axes[0,2].set_title('Horsepower vs Price', fontweight='bold')
axes[0,2].set_xlabel('Horsepower'); axes[0,2].set_ylabel('Price')

axes[1,0].scatter(df['enginesize'], df['price'], alpha=0.5, color='#E67E22', s=30)
axes[1,0].set_title('Engine Size vs Price', fontweight='bold')
axes[1,0].set_xlabel('Engine Size'); axes[1,0].set_ylabel('Price')

axes[1,1].scatter(df['curbweight'], df['price'], alpha=0.5, color='#2ECC71', s=30)
axes[1,1].set_title('Curb Weight vs Price', fontweight='bold')
axes[1,1].set_xlabel('Curb Weight'); axes[1,1].set_ylabel('Price')

axes[1,2].scatter(df['highwaympg'], df['price'], alpha=0.5, color='#1ABC9C', s=30)
axes[1,2].set_title('Highway MPG vs Price', fontweight='bold')
axes[1,2].set_xlabel('Highway MPG'); axes[1,2].set_ylabel('Price')

plt.suptitle('Car Price EDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('carprice_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.1f', cmap='coolwarm',
            square=True, linewidths=0.3, annot_kws={'size': 7})
plt.title('Correlation Heatmap — All Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('carprice_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Train/Test Split & Scaling

In [ ]:
features = [c for c in df.columns if c != 'price']
X = df[features]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Features : {len(features)}")
print(f"Train    : {X_train.shape}")
print(f"Test     : {X_test.shape}")

## 6. Train Multiple Models

In [ ]:
models = {
    'Linear Regression':  LinearRegression(),
    'Ridge':              Ridge(alpha=1.0),
    'Lasso':              Lasso(alpha=1.0),
    'Decision Tree':      DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest':      RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    mae    = mean_absolute_error(y_test, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    r2     = r2_score(y_test, y_pred)
    results[name] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    print(f"{name:<22} R²={r2:.4f}  MAE=${mae:,.0f}  RMSE=${rmse:,.0f}")

## 7. Model Comparison Plot

In [ ]:
res_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
clrs   = plt.cm.viridis(np.linspace(0.2, 0.8, len(res_df)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bars = axes[0].bar(res_df.index, res_df['R2'], color=clrs)
axes[0].set_ylabel('R² Score'); axes[0].set_ylim(0, 1.05)
axes[0].set_title('R² Score Comparison', fontweight='bold')
axes[0].set_xticklabels(res_df.index, rotation=15, ha='right')
for bar, v in zip(bars, res_df['R2']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{v:.3f}', ha='center', fontweight='bold')

axes[1].bar(res_df.index, res_df['MAE']/1000, color=clrs)
axes[1].set_ylabel('MAE ($000s)'); axes[1].set_title('Mean Absolute Error', fontweight='bold')
axes[1].set_xticklabels(res_df.index, rotation=15, ha='right')
plt.tight_layout()
plt.savefig('carprice_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Best Model — Actual vs Predicted

In [ ]:
best_name  = res_df.index[0]
best_model = models[best_name]
y_pred     = best_model.predict(X_test_s)
residuals  = y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, y_pred, alpha=0.6, color='#3498DB', s=30)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Price ($)'); axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'{best_name} — Actual vs Predicted', fontweight='bold')
axes[0].legend()

axes[1].scatter(y_pred, residuals, alpha=0.6, color='#E74C3C', s=30)
axes[1].axhline(0, color='black', lw=1.5)
axes[1].set_xlabel('Predicted Price ($)'); axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Residual Plot', fontweight='bold')
plt.tight_layout()
plt.savefig('carprice_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Best Model : {best_name}")
print(f"R² Score   : {results[best_name]['R2']:.4f}")
print(f"MAE        : ${results[best_name]['MAE']:,.0f}")
print(f"RMSE       : ${results[best_name]['RMSE']:,.0f}")

## 9. Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=True)
    plt.figure(figsize=(10, 6))
    imp.plot(kind='barh', color='#3498DB')
    plt.title(f'Feature Importance — {best_name}', fontsize=13, fontweight='bold')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.savefig('carprice_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

## 10. Predict on a New Car

In [ ]:
# Use mean values from training set for a quick prediction example
sample = X_test.iloc[[0]]
sample_s = scaler.transform(sample)
pred = best_model.predict(sample_s)
actual = y_test.iloc[0]
print(f"Sample car features (first test row):")
print(sample.to_dict('records')[0])
print(f"\nActual Price    : ${actual:,.0f}")
print(f"Predicted Price : ${pred[0]:,.0f}")
print(f"Error           : ${abs(actual - pred[0]):,.0f}")